In [59]:
import pandas as pd, sqlite3

files = [
    "CCDB_Export_13_September_2025_through_October_2025.csv",
    "CCDB_Export_14_November_2025_through_December_2025.csv",
]

df = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)

print(df.shape)              # rows, columns
print(df.columns.tolist())   # column names
print(df.head())             # first 5 rows

con = sqlite3.connect("cfpb.db")
df.to_sql("complaints", con, if_exists="replace", index=False)
con.close()

(2032768, 16)
['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID']
  Date received                                            Product  \
0    2025-09-07  Credit reporting or other personal consumer re...   
1    2025-10-28  Credit reporting or other personal consumer re...   
2    2025-09-13  Credit reporting or other personal consumer re...   
3    2025-09-13  Credit reporting or other personal consumer re...   
4    2025-09-13  Credit reporting or other personal consumer re...   

        Sub-product                                              Issue  \
0  Credit reporting               Incorrect information on your report   
1  Credit reporting   Unable to get your credit report or credit score   
2  Credit reporting  Problem with a company's investigation into an..

In [60]:
print("Rows, columns:", df.shape)
print()
for i, col in enumerate(df.columns, 1):
    print(i, col)

Rows, columns: (2032768, 16)

1 Date received
2 Product
3 Sub-product
4 Issue
5 Sub-issue
6 Consumer complaint narrative
7 Company public response
8 Company
9 State
10 ZIP code
11 Tags
12 Submitted via
13 Date sent to company
14 Company response to consumer
15 Timely response?
16 Complaint ID


In [61]:
print("Rows, columns:", df.shape)
df["Date received"] = pd.to_datetime(df["Date received"])
print(df["Date received"].min(), "to", df["Date received"].max())


Rows, columns: (2032768, 16)
2025-09-01 00:00:00 to 2025-12-31 00:00:00


In [62]:
(df.isna().mean() * 100).round(1).sort_values(ascending=False)


Tags                            97.9
Consumer complaint narrative    85.6
Company public response         41.6
Sub-issue                        1.2
State                            0.1
Sub-product                      0.0
Issue                            0.0
Date received                    0.0
Company                          0.0
Product                          0.0
ZIP code                         0.0
Submitted via                    0.0
Date sent to company             0.0
Company response to consumer     0.0
Timely response?                 0.0
Complaint ID                     0.0
dtype: float64

In [63]:
print("Duplicate Complaint IDs:", df["Complaint ID"].duplicated().sum())

Duplicate Complaint IDs: 0


In [64]:
print(df["Company response to consumer"].value_counts(dropna=False))
print(df["Timely response?"].value_counts(dropna=False))
print(df["Product"].value_counts().head(10))

Company response to consumer
Closed with explanation            1299939
Closed with non-monetary relief     718792
Closed with monetary relief           9247
Untimely response                     4505
In progress                            285
Name: count, dtype: int64
Timely response?
Yes    2024978
No        7790
Name: count, dtype: int64
Product
Credit reporting or other personal consumer reports        1825350
Debt collection                                             107352
Credit card                                                  30637
Checking or savings account                                  26218
Money transfer, virtual currency, or money service           12450
Mortgage                                                      8978
Vehicle loan or lease                                         7700
Student loan                                                  5490
Payday loan, title loan, personal loan, or advance loan       5104
Prepaid card                                  

In [65]:
df.groupby(df["Date received"].dt.to_period("M")).size()


Date received
2025-09    503850
2025-10    519786
2025-11    496658
2025-12    512474
Freq: M, dtype: int64

In [66]:
df.iloc[0]


Date received                                                 2025-09-07 00:00:00
Product                         Credit reporting or other personal consumer re...
Sub-product                                                      Credit reporting
Issue                                        Incorrect information on your report
Sub-issue                                                Account status incorrect
Consumer complaint narrative                                                  NaN
Company public response         Company has responded to the consumer and the ...
Company                                       Experian Information Solutions Inc.
State                                                                          NY
ZIP code                                                                    11580
Tags                                                                          NaN
Submitted via                                                                 Web
Date sent to com

In [67]:
df.iloc[0].astype(str).str[:100]

Date received                                                 2025-09-07 00:00:00
Product                         Credit reporting or other personal consumer re...
Sub-product                                                      Credit reporting
Issue                                        Incorrect information on your report
Sub-issue                                                Account status incorrect
Consumer complaint narrative                                                  NaN
Company public response         Company has responded to the consumer and the ...
Company                                       Experian Information Solutions Inc.
State                                                                          NY
ZIP code                                                                    11580
Tags                                                                          NaN
Submitted via                                                                 Web
Date sent to com

In [68]:
clean = df[df["Company response to consumer"] != "In progress"].copy()

clean["relief"] = clean["Company response to consumer"].isin(
    ["Closed with monetary relief", "Closed with non-monetary relief"]
).astype(int)

clean["Date received"] = pd.to_datetime(clean["Date received"]).dt.strftime("%Y-%m-%d")
clean["month"] = clean["Date received"].str[:7]
clean["Tags"] = clean["Tags"].fillna("None")

con = sqlite3.connect("cfpb.db")
clean.to_sql("complaints_clean", con, if_exists="replace", index=False)

print("Rows in clean table:", len(clean))
print("Overall relief rate:", round(clean["relief"].mean() * 100, 1), "%")
con.close()

Rows in clean table: 2032483
Overall relief rate: 35.8 %


In [69]:
con = sqlite3.connect("cfpb.db")

query = """
SELECT
    Product,
    COUNT(*) AS complaints,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM complaints_clean), 1) AS pct_of_total,
    SUM(relief) AS relief_cases,
    ROUND(100.0 * AVG(relief), 1) AS relief_rate_pct
FROM complaints_clean
GROUP BY Product
ORDER BY complaints DESC;
"""

by_product = pd.read_sql(query, con)
by_product

,Product,complaints,pct_of_total,relief_cases,relief_rate_pct
0,Credit reporting or other personal consumer re...,1825285,89.8,687073,37.6
1,Debt collection,107331,5.3,23795,22.2
2,Credit card,30636,1.5,9311,30.4
3,Checking or savings account,26217,1.3,4485,17.1
4,"Money transfer, virtual currency, or money ser...",12448,0.6,1283,10.3
5,Mortgage,8978,0.4,412,4.6
6,Vehicle loan or lease,7700,0.4,662,8.6
7,Student loan,5296,0.3,113,2.1
8,"Payday loan, title loan, personal loan, or adv...",5104,0.3,273,5.3
9,Prepaid card,2137,0.1,430,20.1


In [70]:
pd.read_sql("""
SELECT Product, COUNT(*) AS in_progress
FROM complaints
WHERE "Company response to consumer" = 'In progress'
GROUP BY Product ORDER BY in_progress DESC
""", sqlite3.connect("cfpb.db"))

,Product,in_progress
0,Student loan,194
1,Credit reporting or other personal consumer re...,65
2,Debt collection,21
3,"Money transfer, virtual currency, or money ser...",2
4,Debt or credit management,1
5,Credit card,1
6,Checking or savings account,1


In [71]:
import pandas as pd, sqlite3
con = sqlite3.connect("cfpb.db")

q1 = """
SELECT Issue, COUNT(*) AS complaints, ROUND(100.0*AVG(relief),1) AS relief_rate_pct
FROM complaints_clean
WHERE Product LIKE 'Credit reporting%'
GROUP BY Issue
ORDER BY complaints DESC;
"""
print(pd.read_sql(q1, con).to_string())

                                                             Issue  complaints  relief_rate_pct
0                             Incorrect information on your report     1090868             37.8
1                                      Improper use of your report      384168             37.3
2  Problem with a company's investigation into an existing problem      334597             38.0
3                    Problem with fraud alerts or security freezes        6553             26.4
4                 Unable to get your credit report or credit score        5667             15.9
5          Credit monitoring or identity theft protection services        2400             31.0
6    Problem with a company's investigation into an existing issue         842             17.1
7           Identity theft protection or other monitoring services         190             16.3


In [72]:
q2 = """
SELECT Company,
       COUNT(*) AS complaints,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM complaints_clean), 1) AS pct_of_total,
       ROUND(100.0 * AVG(relief), 1) AS relief_rate_pct
FROM complaints_clean
GROUP BY Company
HAVING COUNT(*) >= 1000
ORDER BY complaints DESC
LIMIT 20;
"""
print(pd.read_sql(q2, con).to_string())

                                   Company  complaints  pct_of_total  relief_rate_pct
0                            EQUIFAX, INC.      646084          31.8             58.6
1   TRANSUNION INTERMEDIATE HOLDINGS, INC.      616446          30.3             48.9
2      Experian Information Solutions Inc.      510611          25.1              0.1
3        CAPITAL ONE FINANCIAL CORPORATION       10158           0.5             17.4
4                      CBC Companies, Inc.        9266           0.5             65.7
5          Resurgent Capital Services L.P.        8939           0.4             23.3
6                     JPMORGAN CHASE & CO.        8575           0.4             13.1
7                               LEXISNEXIS        8034           0.4             27.2
8                           CITIBANK, N.A.        7376           0.4             47.7
9                    WELLS FARGO & COMPANY        7247           0.4             22.8
10               ENCORE CAPITAL GROUP INC.        7113

In [73]:
q3 = """
SELECT Company,
       month,
       COUNT(*) AS complaints,
       ROUND(100.0 * AVG(relief), 1) AS relief_rate_pct,
       ROUND(100.0 * AVG("Company response to consumer" = 'Closed with explanation'), 1) AS explanation_pct
FROM complaints_clean
WHERE Company IN ('EQUIFAX, INC.',
                  'TRANSUNION INTERMEDIATE HOLDINGS, INC.',
                  'Experian Information Solutions Inc.')
GROUP BY Company, month
ORDER BY Company, month;
"""
print(pd.read_sql(q3, con).to_string())

                                   Company    month  complaints  relief_rate_pct  explanation_pct
0                            EQUIFAX, INC.  2025-09      161574             57.9             42.1
1                            EQUIFAX, INC.  2025-10      167451             60.6             39.4
2                            EQUIFAX, INC.  2025-11      157920             60.6             39.4
3                            EQUIFAX, INC.  2025-12      159139             55.2             44.8
4      Experian Information Solutions Inc.  2025-09      126060              0.1             99.9
5      Experian Information Solutions Inc.  2025-10      130272              0.1             99.9
6      Experian Information Solutions Inc.  2025-11      125933              0.1             99.9
7      Experian Information Solutions Inc.  2025-12      128346              0.2             99.8
8   TRANSUNION INTERMEDIATE HOLDINGS, INC.  2025-09      156792             44.1             55.9
9   TRANSUNION INTER

In [74]:
q4 = """
SELECT Company, Issue,
       COUNT(*) AS complaints,
       ROUND(100.0 * AVG(relief), 1) AS relief_rate_pct
FROM complaints_clean
WHERE Company IN ('EQUIFAX, INC.',
                  'TRANSUNION INTERMEDIATE HOLDINGS, INC.',
                  'Experian Information Solutions Inc.')
  AND Issue IN ('Incorrect information on your report',
                'Improper use of your report',
                'Problem with a company''s investigation into an existing problem')
GROUP BY Company, Issue
ORDER BY Issue, Company;
"""
print(pd.read_sql(q4, con).to_string())

                                  Company                                                            Issue  complaints  relief_rate_pct
0                           EQUIFAX, INC.                                      Improper use of your report      134032             56.4
1     Experian Information Solutions Inc.                                      Improper use of your report       99168              0.1
2  TRANSUNION INTERMEDIATE HOLDINGS, INC.                                      Improper use of your report      134057             48.0
3                           EQUIFAX, INC.                             Incorrect information on your report      381472             60.1
4     Experian Information Solutions Inc.                             Incorrect information on your report      303548              0.1
5  TRANSUNION INTERMEDIATE HOLDINGS, INC.                             Incorrect information on your report      363804             48.0
6                           EQUIFAX, INC.  Probl

In [75]:
con = sqlite3.connect("cfpb.db")

pd.read_sql("""
SELECT Product, COUNT(*) AS complaints, ROUND(100.0*AVG(relief),1) AS relief_rate_pct
FROM complaints_clean GROUP BY Product
""", con).to_csv("summary_by_product.csv", index=False)

pd.read_sql("""
SELECT Company, month, COUNT(*) AS complaints, ROUND(100.0*AVG(relief),1) AS relief_rate_pct
FROM complaints_clean
WHERE Company IN ('EQUIFAX, INC.','TRANSUNION INTERMEDIATE HOLDINGS, INC.','Experian Information Solutions Inc.')
GROUP BY Company, month
""", con).to_csv("summary_bureaus_by_month.csv", index=False)

pd.read_sql("""
SELECT Company, Issue, COUNT(*) AS complaints, ROUND(100.0*AVG(relief),1) AS relief_rate_pct
FROM complaints_clean
WHERE Company IN ('EQUIFAX, INC.','TRANSUNION INTERMEDIATE HOLDINGS, INC.','Experian Information Solutions Inc.')
GROUP BY Company, Issue
""", con).to_csv("summary_bureaus_by_issue.csv", index=False)

print("Done")

Done
